# Tool Registry — Agentic AI Project

A production-style tool management layer for an AI agent.
Demonstrates: Enum, Pydantic BaseModel, Field constraints, computed_field,
@field_validator, @property, __call__, @classmethod factory, model_dump/validate.

**Run cells in order.**

## Imports

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field, ValidationError, computed_field, field_validator
from pprint import pprint


## 1. ToolType (Enum)

Inheriting from `str` means Pydantic serializes it as a plain string in `model_dump()` — no extra config needed.

In [ ]:
class ToolType(str, Enum):
    SEARCH = 'search'
    CALCULATOR = 'calculator'
    API_CALL = 'api_call'

# str inheritance means it serializes cleanly
print(ToolType.SEARCH)          # ToolType.SEARCH
print(ToolType.SEARCH.value)    # search
print(isinstance(ToolType.SEARCH, str))  # True


## 2. ToolConfig (Pydantic BaseModel)

Nested model for per-tool settings. `@field_validator` enforces the `sk-` prefix on API keys.

In [ ]:
class ToolConfig(BaseModel):
    timeout: int = Field(ge=1, le=60)
    max_retries: int = Field(default=3, ge=0, le=5)
    api_key: str | None = None

    @field_validator('api_key')
    @classmethod
    def validate_api_key(cls, value: str | None) -> str | None:
        if value is None:
            return value
        if not value.startswith('sk-'):
            raise ValueError("API key must start with 'sk-'")
        return value

# Valid
cfg = ToolConfig(timeout=20, api_key='sk-abc123')
print(cfg)

# Invalid timeout
try:
    ToolConfig(timeout=100)
except ValidationError as e:
    print('Bad timeout:', e.errors()[0]['msg'])

# Invalid api_key
try:
    ToolConfig(timeout=10, api_key='bad_key')
except ValidationError as e:
    print('Bad api_key:', e.errors()[0]['msg'])


## 3. Tool (Pydantic BaseModel with computed_field)

`@field_validator` lowercases name. `@computed_field` exposes `summary` as a property that also appears in `model_dump()`.

In [ ]:
class Tool(BaseModel):
    name: str = Field(min_length=2)
    tool_type: ToolType
    description: str = 'No description'
    config: ToolConfig
    enabled: bool = True

    @field_validator('name')
    @classmethod
    def lowercase_name(cls, value: str) -> str:
        return value.lower()

    @computed_field
    @property
    def summary(self) -> str:
        return f'{self.name} ({self.tool_type.value}) - {self.description}'

# Name gets lowercased, nested config validated, summary computed
tool = Tool(
    name='Search',
    tool_type=ToolType.SEARCH,
    description='Search the web',
    config=ToolConfig(timeout=20, api_key='sk-abc')
)
print(tool.name)      # search (lowercased)
print(tool.summary)   # computed property
print()
pprint(tool.model_dump())  # summary included in output


## 4. ToolExecutor (__call__ + @property)

Stateful callable — stores the tool and tracks how many times it's been called. `__call__` gives function-like syntax while the object holds state.

In [ ]:
class ToolExecutor:

    def __init__(self, tool: Tool):
        self.tool = tool
        self._execution_count = 0

    def __call__(self, input_data: str) -> str:
        if not self.tool.enabled:
            raise RuntimeError(f"Tool '{self.tool.name}' is disabled.")
        self._execution_count += 1
        return f'[{self.tool.name}] Executed with: {input_data}'

    @property
    def execution_count(self) -> int:
        return self._execution_count

executor = ToolExecutor(tool)
print(executor('Latest AI news'))     # called like a function
print(executor('Pydantic v2 docs'))
print(f'Called {executor.execution_count} times')  # state preserved

# Disable and try
tool.enabled = False
try:
    executor('test')
except RuntimeError as e:
    print(f'Blocked: {e}')
tool.enabled = True  # re-enable for next cells


## 5. ToolRegistry (complete system)

Central registry — stores tools and their dedicated executors. `from_config()` is a `@classmethod` factory using `cls()` instead of hardcoded `ToolRegistry()` for subclass safety.

In [ ]:
class ToolRegistry:

    def __init__(self):
        self.tools: dict[str, Tool] = {}
        self._executors: dict[str, ToolExecutor] = {}

    def register(self, tool: Tool) -> None:
        if tool.name in self.tools:
            raise ValueError(f"Tool '{tool.name}' already registered.")
        self.tools[tool.name] = tool
        self._executors[tool.name] = ToolExecutor(tool)

    def get(self, name: str) -> Tool:
        if name not in self.tools:
            raise KeyError(f"Tool '{name}' not found.")
        return self.tools[name]

    def disable(self, name: str) -> None:
        self.get(name).enabled = False

    def enable(self, name: str) -> None:
        self.get(name).enabled = True

    def execute(self, name: str, input_data: str) -> str:
        tool = self.get(name)
        if not tool.enabled:
            raise RuntimeError(f"Tool '{name}' is disabled.")
        return self._executors[name](input_data)

    def execution_count(self, name: str) -> int:
        if name not in self._executors:
            raise KeyError(f"Tool '{name}' not found.")
        return self._executors[name].execution_count

    def list_enabled(self) -> list[Tool]:
        return [t for t in self.tools.values() if t.enabled]

    def to_schema(self) -> list[dict]:
        return [tool.model_dump() for tool in self.tools.values()]

    @classmethod
    def from_config(cls, data: list[dict]) -> 'ToolRegistry':
        registry = cls()  # cls() not ToolRegistry() — subclass safe
        for item in data:
            tool = Tool.model_validate(item)
            registry.register(tool)
        return registry


## 6. Full Demo

In [ ]:
raw_tools = [
    {
        'name': 'Search',
        'tool_type': 'search',
        'description': 'Search the web',
        'config': {'timeout': 20, 'api_key': 'sk-search'}
    },
    {
        'name': 'Calculator',
        'tool_type': 'calculator',
        'description': 'Perform calculations',
        'config': {'timeout': 10}
    },
    {
        'name': 'WeatherAPI',
        'tool_type': 'api_call',
        'description': 'Weather service',
        'config': {'timeout': 30, 'max_retries': 5, 'api_key': 'sk-weather'}
    }
]

registry = ToolRegistry.from_config(raw_tools)

print('=== Enabled Tools ===')
for tool in registry.list_enabled():
    print(tool.summary)

print('\n=== Execute Tools ===')
print(registry.execute('search', 'Latest AI news'))
print(registry.execute('search', 'Pydantic v2 docs'))
print(f'search called {registry.execution_count("search")} times')

print('\n=== Disable Search ===')
registry.disable('search')
print('Enabled now:', [t.name for t in registry.list_enabled()])

try:
    registry.execute('search', 'test')
except RuntimeError as e:
    print(f'Blocked: {e}')

print('\n=== Schema (for LLM) ===')
pprint(registry.to_schema())

print('\n=== Validation Errors ===')
try:
    ToolConfig(timeout=100)
except ValidationError as e:
    print('timeout=100:', e.errors()[0]['msg'])

try:
    ToolConfig(timeout=10, api_key='bad_key')
except ValidationError as e:
    print('bad api_key:', e.errors()[0]['msg'])

try:
    Tool(name='x', tool_type='search', config={'timeout': 5})
except ValidationError as e:
    print('name too short:', e.errors()[0]['msg'])


---
## Concepts Used

| Concept | Where |
|---|---|
| `Enum` (with `str`) | `ToolType` — clean serialization |
| `BaseModel` | `ToolConfig`, `Tool` |
| `Field()` constraints | timeout, max_retries, name min_length |
| Nested models | `Tool.config` is a `ToolConfig` |
| `@field_validator` | lowercase name, sk- prefix check |
| `@computed_field` + `@property` | `Tool.summary` — appears in model_dump() |
| `__call__` | `ToolExecutor` — stateful callable |
| `@property` | `execution_count` — encapsulated counter |
| `@classmethod` factory | `ToolRegistry.from_config()` using `cls()` |
| `model_dump()` | `to_schema()` for LLM tool definitions |
| `model_validate()` | `from_config()` parsing raw dicts |
| `str \| None` typing | `api_key` optional field |
| `dict[str, Tool]` typing | registry internal store |
